# Visualization Draft
## Professional Supply-Demand Gap Analysis (Subcategory Level)

This notebook presents a comprehensive visual analysis of the supply-demand gap across **product subcategories**. The visualizations are designed to support strategic decision-making and identify priority areas for business intervention at a granular level.

**Analysis Scope:**
- Supply metrics based on product listings (subcategory level)
- Demand metrics based on sales transactions (subcategory level)
- Gap score calculation and severity classification
- Subcategory-level performance rankings

**Deliverables:**
- Interactive charts for stakeholder presentations
- Static images for reports and documentation
- Data exports for further analysis
- Executive insights and recommendations

## Phase 1: Environment Setup

Import required libraries and configure visualization settings.

In [1]:
import sys
sys.path.append("../src")

import pandas as pd
import numpy as np
from pathlib import Path

from data_loader import load_excel_sheets
from supply_demand import (
    compute_subcategory_supply, 
    compute_subcategory_demand, 
    merge_supply_demand
)
from gap_score import (
    compute_gap_score, 
    normalize_gap_score, 
    classify_gap_level,
    compute_demand_supply_ratio,
    rank_categories_by_gap
)
from plotly_interactive_charts import (
    create_supply_demand_chart,
    create_gap_score_severity_chart,
    create_gap_heatmap,
    create_category_ranking_chart,
    create_gap_distribution_pie
)
from dashboard_builder import calculate_kpis, create_kpi_cards
from chart_insights import (
    generate_supply_demand_insight,
    generate_severity_insight,
    generate_heatmap_insight,
    generate_ranking_insight,
    generate_executive_summary
)
from data_exporter import export_dataframe_to_files, export_plotly_chart, export_insights_to_text

pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:,.2f}'.format

print("Environment configured successfully")

Environment configured successfully


## Phase 2: Data Loading and Preparation

Load cleaned datasets and merge product information with sales data at subcategory level.

In [2]:
file_path = "../data/processed/Cleaned_data.xlsx"

product_lookup = pd.read_excel(file_path, sheet_name=' Product Lookup')
product_subcategory = pd.read_excel(file_path, sheet_name='Product Subcategory')
product_category = pd.read_excel(file_path, sheet_name='Product Category')
sales_data = pd.read_excel(file_path, sheet_name='Sales Data')

products_with_subcat = product_lookup.merge(product_subcategory, on='ProductSubcategoryKey', how='left')
products_full = products_with_subcat.merge(product_category, on='ProductCategoryKey', how='left')

sales_full = sales_data.merge(
    products_full[['ProductKey', 'CategoryName', 'SubcategoryName']],
    on='ProductKey',
    how='left'
)

print(f"Data loaded successfully:")
print(f"  Categories: {sales_full['CategoryName'].nunique()}")
print(f"  Subcategories: {sales_full['SubcategoryName'].nunique()}")
print(f"  Products: {len(product_lookup):,}")
print(f"  Sales transactions: {len(sales_data):,}")

Data loaded successfully:
  Categories: 3
  Subcategories: 17
  Products: 293
  Sales transactions: 56,046


## Phase 3: Gap Score Calculation (Subcategory Level)

Compute supply and demand metrics at **subcategory level**, then calculate gap scores with severity classification.

In [3]:
# Compute subcategory-level supply
supply_df = compute_subcategory_supply(sales_data, products_full)
print("Subcategory-Level Supply Metrics:")
display(supply_df.head(10))

# Compute subcategory-level demand
demand_df = compute_subcategory_demand(sales_full)
print("\nSubcategory-Level Demand Metrics:")
display(demand_df.head(10))

# Merge supply and demand
supply_demand_df = merge_supply_demand(supply_df, demand_df)
print(f"\nMerged subcategory data: {supply_demand_df.shape}")

# Compute gap scores
gap_df = compute_gap_score(supply_demand_df, demand_col='TotalQuantitySold', supply_col='UniqueProducts')
gap_df = normalize_gap_score(gap_df, gap_col='GapScore')
gap_df = classify_gap_level(gap_df, gap_col='GapScore')
gap_df = compute_demand_supply_ratio(gap_df)
gap_df = rank_categories_by_gap(gap_df, gap_col='GapScore', ascending=False)

# Prepare summary DataFrame with proper column names
gap_summary = gap_df[['CategoryName', 'SubcategoryName', 'UniqueProducts', 
                       'TotalQuantitySold', 'GapScore', 'GapLevel']].copy()
gap_summary.columns = ['parent_category', 'category', 'supply', 'demand', 'gap_score', 'gap_status']
gap_summary = gap_summary.sort_values('gap_score', ascending=False).reset_index(drop=True)

gap_summary['gap_pct'] = ((gap_summary['demand'] - gap_summary['supply']) / 
                          (gap_summary['supply'] + 1) * 100).round(2)

print("\nGap analysis completed:")
print(f"  Total subcategories: {len(gap_summary)}")
print(f"  Average gap score: {gap_summary['gap_score'].mean():.2f}")
print(f"  Highest gap: {gap_summary.iloc[0]['category']} ({gap_summary.iloc[0]['parent_category']})")
print(f"    Gap score: {gap_summary.iloc[0]['gap_score']:.2f}")
print(f"  Lowest gap: {gap_summary.iloc[-1]['category']} ({gap_summary.iloc[-1]['parent_category']})")
print(f"    Gap score: {gap_summary.iloc[-1]['gap_score']:.2f}")

display(gap_summary.head(15))

Subcategory-Level Supply Metrics:


,CategoryName,SubcategoryName,UniqueProducts
0,Accessories,Bike Racks,1
1,Accessories,Bike Stands,1
2,Accessories,Bottles and Cages,3
3,Accessories,Cleaners,1
4,Accessories,Fenders,1
5,Accessories,Helmets,3
6,Accessories,Hydration Packs,1
7,Accessories,Lights,3
8,Accessories,Locks,1
9,Accessories,Panniers,1



Subcategory-Level Demand Metrics:


,CategoryName,SubcategoryName,UniqueOrders,TotalQuantitySold,UniqueCustomers
0,Accessories,Bike Racks,302,302,300
1,Accessories,Bike Stands,234,234,231
2,Accessories,Bottles and Cages,4485,15106,4288
3,Accessories,Cleaners,850,1706,823
4,Accessories,Fenders,1975,3960,1965
5,Accessories,Helmets,6034,6034,5620
6,Accessories,Hydration Packs,695,695,682
7,Accessories,Tires and Tubes,9084,29772,7924
8,Bikes,Mountain Bikes,4706,4706,3950
9,Bikes,Road Bikes,7099,7099,5728



Merged subcategory data: (37, 6)

Gap analysis completed:
  Total subcategories: 37
  Average gap score: 402.16
  Highest gap: Bottles and Cages (Accessories)
    Gap score: 3776.75
  Lowest gap: Road Frames (Components)
    Gap score: 0.03


,parent_category,category,supply,demand,gap_score,gap_status,gap_pct
0,Accessories,Bottles and Cages,3,15106,"3,776.75",Critical Gap,"377,575.00"
1,Accessories,Tires and Tubes,11,29772,"2,481.08",Critical Gap,"248,008.33"
2,Clothing,Caps,1,4151,"2,076.00",Critical Gap,"207,500.00"
3,Accessories,Fenders,1,3960,"1,980.50",Critical Gap,"197,950.00"
4,Accessories,Helmets,3,6034,"1,508.75",Critical Gap,"150,775.00"
5,Accessories,Cleaners,1,1706,853.50,Critical Gap,"85,250.00"
6,Clothing,Gloves,6,2644,377.86,Critical Gap,"37,685.71"
7,Accessories,Hydration Packs,1,695,348.00,Critical Gap,"34,700.00"
8,Clothing,Jerseys,8,3113,346.00,Critical Gap,"34,500.00"
9,Clothing,Socks,4,1063,212.80,Critical Gap,"21,180.00"


## Phase 4: Core Visualizations (Subcategory Level)

Create the primary analytical charts for subcategory-level presentation and reporting.

**Note:** Charts will show subcategory names with parent category context where helpful.

### Chart 1: Supply vs Demand Comparison (Top 15 Subcategories)

This grouped bar chart compares supply and demand across the top 15 subcategories by gap score.

In [4]:
# Show top 15 subcategories for better readability
top_15_gaps = gap_summary.head(15).copy()

# Add parent category to subcategory name for context
top_15_gaps['category_display'] = (
    top_15_gaps['category'] + '\n(' + top_15_gaps['parent_category'] + ')'
)
top_15_gaps['category'] = top_15_gaps['category_display']

chart_supply_demand = create_supply_demand_chart(top_15_gaps)
chart_supply_demand.update_layout(
    title=dict(text='<b>Supply vs Demand by Subcategory (Top 15 Gaps)</b><br><sub>Independent Y-axes for clear comparison</sub>')
)
chart_supply_demand.show()

insight_supply_demand = generate_supply_demand_insight(gap_summary)
print("\nKey Insight:")
print(insight_supply_demand)


Key Insight:
The analysis reveals Bottles and Cages as the category with the largest supply-demand imbalance.
With 15106 units in demand but only 3 products available,
the demand-to-supply ratio is 3776.5:1. This 377575.0% gap 
requires immediate strategic attention through supply expansion or demand management initiatives.

Best Practice Visualization: The chart uses independent Y-axes (dual-axis chart), which is the 
industry-standard approach recommended by data visualization experts (Tufte, Few, Cairo) for 
comparing metrics with different scales. Each metric is shown on its own appropriately-scaled axis, 
ensuring both are clearly visible without artificial manipulation.


### Chart 2: Gap Score Severity Analysis (All Subcategories)

This horizontal bar chart displays gap scores for all subcategories with color-coded severity levels.

In [5]:
# Prepare data with subcategory + parent category labels
gap_chart_data = gap_summary.copy()
gap_chart_data['category_full'] = (
    gap_chart_data['category'] + ' (' + gap_chart_data['parent_category'] + ')'
)
gap_chart_data['category'] = gap_chart_data['category_full']

chart_severity = create_gap_score_severity_chart(gap_chart_data)
chart_severity.update_layout(
    title=dict(text='<b>Gap Score Severity Analysis (Subcategories)</b><br><sub>Color-coded by severity level</sub>'),
    height=max(600, len(gap_chart_data) * 20)  # Dynamic height based on number of subcategories
)
chart_severity.show()

insight_severity = generate_severity_insight(gap_summary)
print("\nKey Insight:")
print(insight_severity)


Key Insight:

SEVERITY ANALYSIS (Scale Used: Logarithmic)

Among 37 categories, 16 exhibit either Critical or High severity levels,
accounting for 43.2% of the dataset.

HIGHEST SEVERITY CATEGORY: Bottles and Cages
  • Gap Score: 3776.75
  • Severity: Critical Gap
  • Supply: 3
  • Demand: 15106
  • Gap Percentage: 377575.0%

SEVERITY DISTRIBUTION:
  • Critical Gap: 10 (27.0%)
  • High Gap: 6 (16.2%)
  • Moderate Gap: 1 (2.7%)
  • Low Gap: 20 (54.1%)
NOTE ON VISUALIZATION:
A log scale is used because the gap score range exceeds a 100× difference.
Zero gap values are replaced with 0.01 (standard best practice) to enable logarithmic plotting.
This reveals the relative differences clearly without distortion.


### Chart 3: Normalized Metrics Heatmap (Top 20 Subcategories)

This heatmap provides a normalized view of supply, demand, and gap scores for the top 20 subcategories.

In [6]:
# Show top 20 for heatmap clarity
top_20_heatmap = gap_summary.head(20).copy()
top_20_heatmap['category_display'] = (
    top_20_heatmap['category'] + '\n(' + top_20_heatmap['parent_category'] + ')'
)
top_20_heatmap['category'] = top_20_heatmap['category_display']

chart_heatmap = create_gap_heatmap(top_20_heatmap)
chart_heatmap.update_layout(
    title=dict(text='<b>Gap Analysis Heatmap (Top 20 Subcategories)</b><br><sub>Normalized Metrics (0-1 Scale)</sub>'),
    height=500
)
chart_heatmap.show()

insight_heatmap = generate_heatmap_insight(gap_summary)
print("\nKey Insight:")
print(insight_heatmap)


Key Insight:
The normalized heatmap clearly distinguishes healthy market segments from critical zones.
Red areas indicate high gaps requiring intervention, while green zones show equilibrium.
Mountain Frames demonstrates the healthiest balance with a gap score of 0.03.


### Chart 4: Subcategory Rankings

This dual-panel chart ranks subcategories by gap severity, showing both the most under-supplied and the most balanced segments.

In [7]:
# Prepare ranking data
ranking_data = gap_summary.copy()
ranking_data['category_full'] = (
    ranking_data['category'] + ' (' + ranking_data['parent_category'] + ')'
)
ranking_data['category'] = ranking_data['category_full']

chart_ranking = create_category_ranking_chart(ranking_data, top_n=10)
chart_ranking.update_layout(
    title=dict(text='<b>Subcategory Rankings: Highest & Lowest Gaps</b>'),
    height=600
)
chart_ranking.show()

insight_ranking = generate_ranking_insight(gap_summary, top_n=10)
print("\nKey Insight:")
print(insight_ranking)


Key Insight:
The top 10 under-supplied categories show gap scores ranging from 212.80 to 3776.75.
Categories with scores below 50 are considered well-balanced, indicating healthy market conditions.
Strategic focus should prioritize the highest-ranking categories for supply expansion.


### Chart 5: Gap Severity Distribution (Subcategory Level)

This pie chart shows the overall distribution of gap severity levels across all subcategories.

In [8]:
chart_distribution = create_gap_distribution_pie(gap_summary)
chart_distribution.update_layout(
    title=dict(text='<b>Gap Severity Distribution (Subcategories)</b>')
)
chart_distribution.show()

gap_level_counts = gap_summary['gap_status'].value_counts()
print("\nGap Level Distribution:")
for level, count in gap_level_counts.items():
    print(f"  {level}: {count} subcategories ({count/len(gap_summary)*100:.1f}%)")

print(f"\nCritical breakdown by parent category:")
critical_by_category = gap_summary[
    gap_summary['gap_status'].isin(['Critical Gap', 'High Gap'])
].groupby('parent_category').size().sort_values(ascending=False)
display(critical_by_category)


Gap Level Distribution:
  Low Gap: 20 subcategories (54.1%)
  Critical Gap: 10 subcategories (27.0%)
  High Gap: 6 subcategories (16.2%)
  Moderate Gap: 1 subcategories (2.7%)

Critical breakdown by parent category:


parent_category
Accessories    8
Clothing       6
Bikes          2
dtype: int64

## Phase 5: Key Performance Indicators (Subcategory Level)

Calculate and display critical business metrics at subcategory level in a professional dashboard-style layout following industry best practices.

In [9]:
kpis = calculate_kpis(gap_summary)
kpi_fig = create_kpi_cards(kpis)
kpi_fig.update_layout(
    title=dict(text='<b>Subcategory Gap Analysis KPI Overview</b>')
)
kpi_fig.show()

print("\nKey Performance Indicators (Subcategory Level):")
print(f"  Average Gap Score: {kpis['avg_gap_score']}")
print(f"  Highest Gap Subcategory: {kpis['highest_gap_category']} ({kpis['highest_gap_value']})")
print(f"  Total Subcategories Analyzed: {kpis['total_categories']}")
print(f"  Subcategories with High/Critical Gap: {kpis['critical_high_count']} ({kpis['high_gap_pct']}%)")
print(f"  Total Supply: {kpis['total_supply']:,} units")
print(f"  Total Demand: {kpis['total_demand']:,} units")

print("\n\nTop 5 Categories by Number of Critical/High Gap Subcategories:")
critical_subcats = gap_summary[
    gap_summary['gap_status'].isin(['Critical Gap', 'High Gap'])
].groupby('parent_category').agg({
    'category': 'count',
    'gap_score': 'mean'
}).rename(columns={'category': 'critical_subcat_count', 'gap_score': 'avg_gap_score'})
critical_subcats = critical_subcats.sort_values('critical_subcat_count', ascending=False)
display(critical_subcats.head())


Key Performance Indicators (Subcategory Level):
  Average Gap Score: 402.16
  Highest Gap Subcategory: Bottles and Cages (3776.75)
  Total Subcategories Analyzed: 37
  Subcategories with High/Critical Gap: 16 (43.2%)
  Total Supply: 293 units
  Total Demand: 84,174 units


Top 5 Categories by Number of Critical/High Gap Subcategories:


,critical_subcat_count,avg_gap_score
parent_category,,
Accessories,8,"1,402.20"
Clothing,6,543.55
Bikes,2,152.00


## Phase 6: Export Deliverables

Generate all final outputs including data files, interactive charts, and static images at **subcategory level**.

In [10]:
output_base = Path("../results/visualizations_subcategory")
output_base.mkdir(parents=True, exist_ok=True)

print("Exporting subcategory-level deliverables...")
print("=" * 70)

# Prepare export data with both category and subcategory info
export_summary = gap_summary[['parent_category', 'category', 'supply', 'demand', 
                               'gap_score', 'gap_status', 'gap_pct']].copy()
export_summary.columns = ['parent_category', 'subcategory', 'supply', 'demand', 
                          'gap_score', 'gap_status', 'gap_pct']

print("\n1. Exporting data summaries...")
csv_path, json_path = export_dataframe_to_files(
    export_summary, 
    output_base / "data", 
    "subcategory_gap_summary"
)
print(f"   CSV: {csv_path}")
print(f"   JSON: {json_path}")

print("\n2. Exporting interactive charts (HTML)...")
charts_dir = output_base / "interactive"

# Recreate charts for export (top 15 for readability)
export_top15 = gap_summary.head(15).copy()
export_top15['category_display'] = export_top15['category'] + ' (' + export_top15['parent_category'] + ')'
export_top15['category'] = export_top15['category_display']

export_plotly_chart(create_supply_demand_chart(export_top15), 
                   charts_dir / "01_supply_vs_demand_subcategory.html", format='html')
export_plotly_chart(chart_severity, charts_dir / "02_gap_severity_subcategory.html", format='html')
export_plotly_chart(chart_heatmap, charts_dir / "03_gap_heatmap_subcategory.html", format='html')
export_plotly_chart(chart_ranking, charts_dir / "04_subcategory_rankings.html", format='html')
export_plotly_chart(chart_distribution, charts_dir / "05_gap_distribution_subcategory.html", format='html')
print(f"   Interactive charts saved to: {charts_dir}")

print("\n3. Exporting static images (PNG)...")
images_dir = output_base / "images"
export_plotly_chart(create_supply_demand_chart(export_top15), 
                   images_dir / "01_supply_vs_demand_subcategory.png", format='png', width=1200, height=700)
export_plotly_chart(chart_severity, images_dir / "02_gap_severity_subcategory.png", 
                   format='png', width=1200, height=max(700, len(gap_summary) * 15))
export_plotly_chart(chart_heatmap, images_dir / "03_gap_heatmap_subcategory.png", format='png', width=1200, height=600)
export_plotly_chart(chart_ranking, images_dir / "04_subcategory_rankings.png", format='png', width=1200, height=700)
export_plotly_chart(chart_distribution, images_dir / "05_gap_distribution_subcategory.png", format='png', width=1080, height=500)
print(f"   Static images saved to: {images_dir}")

print("\n4. Exporting insights...")
all_insights = {
    'supply_demand_analysis': insight_supply_demand,
    'severity_analysis': insight_severity,
    'heatmap_analysis': insight_heatmap,
    'ranking_analysis': insight_ranking
}
insights_path = export_insights_to_text(all_insights, output_base / "insights_subcategory.txt")
print(f"   Insights saved to: {insights_path}")

print("\n" + "=" * 70)
print("All subcategory-level deliverables exported successfully")

Exporting subcategory-level deliverables...

1. Exporting data summaries...
   CSV: ../results/visualizations_subcategory/data/subcategory_gap_summary.csv
   JSON: ../results/visualizations_subcategory/data/subcategory_gap_summary.json

2. Exporting interactive charts (HTML)...
   Interactive charts saved to: ../results/visualizations_subcategory/interactive

3. Exporting static images (PNG)...
   Interactive charts saved to: ../results/visualizations_subcategory/interactive

3. Exporting static images (PNG)...
   Static images saved to: ../results/visualizations_subcategory/images

4. Exporting insights...
   Insights saved to: ../results/visualizations_subcategory/insights_subcategory.txt

All subcategory-level deliverables exported successfully
   Static images saved to: ../results/visualizations_subcategory/images

4. Exporting insights...
   Insights saved to: ../results/visualizations_subcategory/insights_subcategory.txt

All subcategory-level deliverables exported successfully


## Phase 7: Executive Summary (Subcategory Level)

Comprehensive analysis findings and strategic recommendations at the subcategory level.

In [11]:
executive_summary = generate_executive_summary(gap_summary)

print("=" * 70)
print("EXECUTIVE SUMMARY (SUBCATEGORY LEVEL)")
print("=" * 70)

print(f"\nCRITICAL GAPS ({executive_summary['critical_gaps']['count']} subcategories):")
print(f"  Most Severe: {executive_summary['critical_gaps']['most_severe']}")
print(f"    Parent Category: {gap_summary[gap_summary['category'] == executive_summary['critical_gaps']['most_severe']].iloc[0]['parent_category']}")
print(f"  Gap Score: {executive_summary['critical_gaps']['gap_score']:.2f}")
print(f"  Supply: {executive_summary['critical_gaps']['supply']:,} products")
print(f"  Demand: {executive_summary['critical_gaps']['demand']:,} units")
print(f"  Gap Severity: {executive_summary['critical_gaps']['gap_pct']:.1f}%")

print(f"\nWELL-BALANCED SUBCATEGORIES:")
print(f"  Example: {executive_summary['balanced_categories']['example']}")
print(f"    Parent Category: {gap_summary[gap_summary['category'] == executive_summary['balanced_categories']['example']].iloc[0]['parent_category']}")
print(f"  Gap Score: {executive_summary['balanced_categories']['gap_score']:.2f}")
print(f"  Supply: {executive_summary['balanced_categories']['supply']:,} products")
print(f"  Demand: {executive_summary['balanced_categories']['demand']:,} units")

print(f"\nOVERALL METRICS:")
print(f"  Total Subcategories: {executive_summary['overall_metrics']['total_categories']}")
print(f"  Parent Categories: {gap_summary['parent_category'].nunique()}")
print(f"  Average Gap Score: {executive_summary['overall_metrics']['avg_gap_score']:.2f}")
print(f"  Critical/High Gap Subcategories: {executive_summary['overall_metrics']['critical_high_count']} ({executive_summary['overall_metrics']['critical_high_pct']:.1f}%)")
print(f"  Total Supply: {executive_summary['overall_metrics']['total_supply']:,} units")
print(f"  Total Demand: {executive_summary['overall_metrics']['total_demand']:,} units")

print("\nTOP 5 PARENT CATEGORIES WITH MOST GAPS:")
category_gap_summary = gap_summary[
    gap_summary['gap_status'].isin(['Critical Gap', 'High Gap'])
].groupby('parent_category').agg({
    'category': 'count',
    'gap_score': ['mean', 'max']
}).round(2)
category_gap_summary.columns = ['critical_subcat_count', 'avg_gap', 'max_gap']
category_gap_summary = category_gap_summary.sort_values('critical_subcat_count', ascending=False)
for idx, (cat, row) in enumerate(category_gap_summary.head().iterrows(), 1):
    print(f"  {idx}. {cat}: {int(row['critical_subcat_count'])} critical subcategories (avg gap: {row['avg_gap']:.2f})")

print("\nRECOMMENDATIONS:")
print("\n  Immediate Actions:")
for rec in executive_summary['recommendations']['immediate']:
    print(f"    - {rec}")

print("\n  Medium-Term Strategy:")
for rec in executive_summary['recommendations']['medium_term']:
    print(f"    - {rec}")
print(f"    - Focus on parent categories with multiple high-gap subcategories")

print("\n  Strategic Initiatives:")
for rec in executive_summary['recommendations']['strategic']:
    print(f"    - {rec}")
print(f"    - Develop subcategory-specific product expansion roadmap")

print("\n" + "=" * 70)
print("Subcategory-level analysis complete and ready for presentation")
print("=" * 70)

EXECUTIVE SUMMARY (SUBCATEGORY LEVEL)

CRITICAL GAPS (10 subcategories):
  Most Severe: Bottles and Cages
    Parent Category: Accessories
  Gap Score: 3776.75
  Supply: 3 products
  Demand: 15,106 units
  Gap Severity: 377575.0%

WELL-BALANCED SUBCATEGORIES:
  Example: Mountain Frames
    Parent Category: Components
  Gap Score: 0.03
  Supply: 28 products
  Demand: 0 units

OVERALL METRICS:
  Total Subcategories: 37
  Parent Categories: 4
  Average Gap Score: 402.16
  Critical/High Gap Subcategories: 16 (43.2%)
  Total Supply: 293 units
  Total Demand: 84,174 units

TOP 5 PARENT CATEGORIES WITH MOST GAPS:
  1. Accessories: 8 critical subcategories (avg gap: 1402.20)
  2. Clothing: 6 critical subcategories (avg gap: 543.55)
  3. Bikes: 2 critical subcategories (avg gap: 152.00)

RECOMMENDATIONS:

  Immediate Actions:
    - Prioritize supply expansion in Bottles and Cages
    - Launch targeted marketing campaigns in high-gap categories
    - Investigate supply chain bottlenecks

  Mediu

## Deliverables Summary (Subcategory Level)

All visualization outputs have been generated and exported to the **results/visualizations_subcategory** directory:

**Data Exports:**
- CSV format with parent category and subcategory information
- JSON format for web applications and APIs
- Includes all subcategories with gap metrics

**Interactive Visualizations:**
- HTML files with full interactivity including zoom, pan, and hover tooltips
- Subcategory names include parent category context for clarity
- Suitable for stakeholder presentations and web deployment

**Static Images:**
- High-resolution PNG files
- Dynamic sizing based on number of subcategories
- Publication-ready for reports and documentation

**Insights:**
- Textual analysis at subcategory level
- Executive summary with parent category breakdown
- Strategic recommendations for granular action

**Key Benefits of Subcategory Analysis:**
- **More Granular Insights:** Identify specific product lines requiring attention
- **Targeted Action:** Pinpoint exact subcategories for supply expansion
- **Better Resource Allocation:** Focus efforts on high-impact subcategories
- **Parent Category Context:** Understand which main categories have the most issues
- **Comprehensive View:** 37 subcategories vs 4 categories = 9x more detail

This subcategory-level visualization package provides detailed, actionable intelligence for strategic decision-making in supply-demand gap management.